# Circuit matrices with KLU

A circuit simulator such as SPICE writes Kirchhoff's laws for every node, with
modified nodal analysis, and solves the nonlinear system by Newton's method.
Every Newton step, at every time step, factors a Jacobian with the same sparsity
pattern and new values. Circuit matrices are unsymmetric, very sparse, and
highly *reducible*: permuted to block triangular form, most of their diagonal
blocks are single entries. Only the diagonal blocks need an LU factorization.

KLU (Davis and Palamadai Natarajan, 2010) is built for exactly this:

- **Block triangular form (BTF):** it permutes the matrix to block upper
  triangular form, with the Dulmage–Mendelsohn decomposition.
- **Each diagonal block:** it orders the block with AMD and factors it with
  Gilbert–Peierls LU.
- **`refactor`:** a new matrix with the same pattern reuses all of that
  analysis, and the pivot sequence too.

This notebook runs MTL5's native KLU on a real circuit matrix, `adder_dcop_01`.
It measures the structure KLU finds and what it saves, then the case `refactor`
exists for, and the case where `refactor` gives a wrong answer.

**Uses:** `mtl5.sparse.klu` and its `.nblocks`, `.block_starts`, `.B`,
`.block(i)`, `.solve` and `.refactor`; `mtl5.sparse.splu`; `mtl5.io.mm_read`.

**From C++:** [`phase15_sparse_direct/circuit_matrix_klu.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase15_sparse_direct/circuit_matrix_klu.cpp).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from matplotlib.colors import ListedColormap

import mtl5
import mtl5.sparse as ms

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)


def spy(ax, S, title, color=SERIES[0], pixels=300):
    # The stored entries (explicit zeros included) on a grid of at most
    # pixels x pixels cells; a cell is colored if it holds any entry. Drawing one
    # cell per matrix entry would drop entries when the image is scaled down.
    S = sp.coo_matrix(S)
    bins = min(S.shape[0], pixels)
    counts, _, _ = np.histogram2d(S.row, S.col, bins=bins, range=[[0, S.shape[0]], [0, S.shape[1]]])
    extent = (-0.5, S.shape[1] - 0.5, S.shape[0] - 0.5, -0.5)  # in matrix coordinates
    ax.imshow(
        counts > 0, cmap=ListedColormap([SURFACE, color]), interpolation="none", extent=extent
    )
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def relres(S, x, b):
    # ||b - A x||_inf / ||b||_inf
    return np.abs(b - S @ x).max() / np.abs(b).max()


def best_ms(fn, repeats=5):
    best = float("inf")
    for _ in range(repeats):
        t = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t)
    return best * 1e3

## The C++ example's synthetic circuit

With no file given, the C++ program builds a 6 × 6 matrix of two decoupled
3 × 3 blocks, with one coupling entry from the first block to the second. BTF
finds the two blocks. `.B` is the permuted, row-scaled matrix KLU works on:
block upper triangular, with the coupling above the diagonal blocks, where the
block back-substitution uses it as it stands.

In [ ]:
S6 = sp.csr_matrix(
    np.array(
        [
            [3, -1, 0, 0, 0, -2],
            [-1, 3, -1, 0, 0, 0],
            [0, -1, 3, 0, 0, 0],
            [0, 0, 0, 4, -1, 0],
            [0, 0, 0, -1, 4, -1],
            [0, 0, 0, 0, -1, 4],
        ],
        dtype=float,
    )
)
k6 = ms.klu(ms.from_scipy(S6))
print(f"{k6.nblocks} blocks, block_starts {np.asarray(k6.block_starts).tolist()}")
print(np.round(ms.to_scipy(k6.B).toarray(), 3))

b6 = S6 @ np.ones(6)
assert k6.nblocks == 2
assert relres(S6, k6.solve(b6).to_numpy(), b6) < 1e-15

## A real circuit: `adder_dcop_01`

`adder_dcop_01` comes from Sandia National Laboratories' circuit simulations
(R. Hoekstra, 2003). It is the Jacobian of a DC operating point solve for an
adder circuit: 1813 unknowns, and 11,156 entries. The SuiteSparse Matrix
Collection holds it as `Sandia/adder_dcop_01`, and a copy is in
`notebooks/data`. Its pattern is unsymmetric, and its condition number is about
3 × 10⁷.

In [ ]:
A = mtl5.io.mm_read("../data/adder_dcop_01.mtx")
S = ms.to_scipy(A).tocsr()
n = A.shape[0]
unsym = int(((S != 0) != (S.T != 0)).nnz)
cond = np.linalg.cond(S.toarray())
print(
    f"n = {n}, nnz = {A.nnz:,}, entries without a transposed partner: {unsym:,}, cond = {cond:.1e}"
)

assert (n, A.nnz) == (1813, 11156)
assert 1e7 < cond < 1e8

## The block triangular form

KLU finds 469 diagonal blocks. 257 of them are single entries, which need no
factorization at all: back-substitution divides by them. The largest block
has 109 rows. In B the blocks sit along the diagonal, and every other entry
lies above them.

In [ ]:
k = ms.klu(A)
starts = np.asarray(k.block_starts)
sizes = np.diff(starts)
B = ms.to_scipy(k.B).tocsr()
block_of = np.searchsorted(starts, np.arange(n), side="right") - 1
r, c = B.nonzero()
above = int((block_of[c] > block_of[r]).sum())
print(f"{k.nblocks} blocks: {(sizes == 1).sum()} of size 1, largest {sizes.max()}")
print(f"entries above the diagonal blocks: {above:,}, below: {(block_of[c] < block_of[r]).sum()}")

fig, axes = plt.subplots(
    1, 3, figsize=(11, 3.9), dpi=150, gridspec_kw={"width_ratios": [1, 1, 1.15]}
)
spy(axes[0], S, f"A\nnz = {A.nnz:,}", color="#8a8984")
spy(axes[1], B, f"B = P (R A) Q, {k.nblocks} blocks", color=SERIES[0])
for s0, s1 in zip(starts[:-1], starts[1:]):
    if s1 - s0 >= 8:  # the larger blocks; the rest are too small to see
        axes[1].add_patch(
            plt.Rectangle(
                (s0 - 0.5, s0 - 0.5),
                s1 - s0,
                s1 - s0,
                fill=False,
                edgecolor=SERIES[1],
                linewidth=0.6,
            )
        )
values, counts = np.unique(sizes, return_counts=True)
axes[2].bar(range(len(values)), counts, color=SERIES[0])
axes[2].set_xticks(range(len(values)), values, rotation=90, fontsize=7)
axes[2].set_yscale("log")
axes[2].set_xlabel("block size")
axes[2].set_ylabel("blocks")
axes[2].set_title("Diagonal blocks by size", fontsize=9)
plt.tight_layout()
plt.show()

assert k.nblocks == 469 and (sizes == 1).sum() == 257 and sizes.max() == 109
assert (block_of[c] < block_of[r]).sum() == 0  # block upper triangular

## What KLU stores, against one LU of the whole matrix

KLU stores the L and U factors of each diagonal block, and the entries above
the blocks as they are. `splu` factors the whole matrix with one ordering. Its
best ordering here is AMD, and it still stores more than three times as much
as KLU. Both counts leave out the unit diagonal of L.

In [ ]:
klu_stored = k.nnz - n + above
print(f"{'':>12} {'stored':>8}")
print(f"{'KLU':>12} {klu_stored:8,}   (factors {k.nnz - n:,} + above the blocks {above:,})")
stored = {}
for o in ms.orderings():
    stored[o] = ms.splu(A, ordering=o).nnz - n
    print(f"{'splu ' + o:>12} {stored[o]:8,}")

best = min(stored, key=stored.get)
assert best == "amd"
assert klu_stored < stored["amd"] / 3

## The solve

The right-hand side is b = A · 1, as in the C++ program, so the exact solution
is all ones. KLU's residual is at rounding level. The error in x is larger, by
up to the condition number, and it agrees with SciPy's SuperLU, which stands
in for the C++ program's optional cross-check against SuiteSparse's KLU.

In [ ]:
b = S @ np.ones(n)
x = k.solve(b).to_numpy()
x_superlu = spla.spsolve(S.tocsc(), b)
for name, xs in (("KLU", x), ("SuperLU", x_superlu)):
    err = np.abs(xs - 1).max()
    print(f"{name + ':':<8} relative residual {relres(S, xs, b):.1e}, max |x - 1| = {err:.1e}")

assert relres(S, x, b) < 1e-13
assert np.abs(x - 1).max() < 1e-8  # cond(A) * eps is about 1e-8
assert np.abs(x - x_superlu).max() < 1e-8

## `refactor`: the next Newton step

In a simulation the next matrix has the same pattern and slightly different
values: device conductances change with the operating point. Here that is
modelled by changing every value by about 1%, with a fixed seed. `refactor`
skips the block triangular form, the orderings and the pivot search, and redoes
only the arithmetic.

A refactorization is only as good as the old pivot sequence is for the new
values. KLU's check for that is the pivot growth, max |U| / max |block| over
the diagonal blocks. It is near 1 when the pivots are still good, and it is
computed below from `.block(i)`.

The times are this machine's and are not checked.

In [ ]:
def pivot_growth(f):
    # max over diagonal blocks of max|U_i| / max|B_i|
    Bf = ms.to_scipy(f.B).tocsr()
    g = 0.0
    for i in range(f.nblocks):
        blk = f.block(i)
        lo, hi = blk["start"], blk["stop"]
        g = max(g, abs(ms.to_scipy(blk["U"])).max() / abs(Bf[lo:hi, lo:hi]).max())
    return g


rng = np.random.default_rng(147)
S_next = S.copy()
S_next.data *= 1 + 0.01 * rng.standard_normal(S_next.nnz)
A_next = ms.from_scipy(S_next)
b_next = S_next @ np.ones(n)

k.refactor(A_next)
r_next = relres(S_next, k.solve(b_next).to_numpy(), b_next)
g_next = pivot_growth(k)
print(f"after refactor: relative residual {r_next:.1e}, pivot growth {g_next:.2f}")

t_factor = best_ms(lambda: ms.klu(A_next))
t_refactor = best_ms(lambda: k.refactor(A_next))
t_splu = best_ms(lambda: ms.splu(A_next, ordering="amd"))
print(f"klu {t_factor:.2f} ms, refactor {t_refactor:.2f} ms, splu(amd) {t_splu:.2f} ms")

assert r_next < 1e-13
assert g_next < 10

## When the old pivots don't fit

`adder_dcop_02` is the next matrix of the same Sandia sequence. Its pattern has
90 more entries, all of them above the diagonal blocks, where the solve uses
them as they stand, so `refactor` accepts it. Nearly every value is different,
though, and the condition number is about 3 × 10¹⁴. The old pivot sequence
meets tiny pivots, and the result is useless. Nothing raises an error: MTL5's
`refactor` raises only on a pivot that is exactly zero. The residual shows the
failure, and so does the pivot growth.

So check a refactorization before using its answer, and factor afresh when the
check fails. SuiteSparse's KLU provides `klu_rgrowth` and `klu_condest` for
this check, and MTL5's native KLU doesn't yet, so the residual is the check
here. A fresh `klu` chooses new pivots and solves `adder_dcop_02` to rounding
level.

In [ ]:
A2 = mtl5.io.mm_read("../data/adder_dcop_02.mtx")
S2 = ms.to_scipy(A2).tocsr()
b2 = S2 @ np.ones(n)
cond2 = np.linalg.cond(S2.toarray())
print(f"adder_dcop_02: nnz {A2.nnz:,} ({A2.nnz - A.nnz} more), cond {cond2:.1e}")


def solve_next(f, A_new, S_new, b_new, tol=1e-10):
    # Refactor, check the residual, and fall back to a fresh factorization.
    try:
        f.refactor(A_new)
        x = f.solve(b_new).to_numpy()
        if relres(S_new, x, b_new) <= tol:
            return f, x, "refactor"
    except RuntimeError:  # a reused pivot that is exactly zero
        pass
    f = ms.klu(A_new)
    return f, f.solve(b_new).to_numpy(), "fresh factorization"


k = ms.klu(A)
k.refactor(A2)
r_bad, g_bad = relres(S2, k.solve(b2).to_numpy(), b2), pivot_growth(k)
print(f"refactor:            relative residual {r_bad:.1e}, pivot growth {g_bad:.1e}")

k, x2, how = solve_next(ms.klu(A), A2, S2, b2)
r2, g2 = relres(S2, x2, b2), pivot_growth(k)
print(f"{how + ':':<20} relative residual {r2:.1e}, pivot growth {g2:.2f}")

assert r_bad > 1e-3 and g_bad > 1e6
assert how == "fresh factorization"
assert r2 < 1e-13
assert g2 < 10

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `io::mm_read<double>(file)` | `mtl5.io.mm_read(path)` |
| `native_klu_factor(A)` | `ms.klu(A)` |
| `fac.nblocks()` | `.nblocks`, with `.block_starts`, `.B`, `.row_perm`, `.col_perm` and `.block(i)` to look inside |
| `fac.solve(x, b)` | `x = fac.solve(b)` |
| `native_klu_refactor(A2, fac)` | `fac.refactor(A2)` |
| `interface::klu_solve(A, x, b)`, SuiteSparse's KLU if built with it | SciPy's `spsolve` (SuperLU) for the cross-check |

The C++ program reads a matrix fetched by `fetch_circuit_matrices.sh`, by
default `rajat30` (643,994 unknowns), and times the factorization. Here the
matrices are small enough to keep in the repository, so the notebook needs no
download. What is checked is the structure, the stored entries and the
accuracy, not the times. `refactor` is added, with the check that tells
whether its answer can be used.